# CICCADA Data Calc Write Stage 2: Conformance Table Builders

Builds `conformance_voltvar_v2`, `conformance_voltwatt_v2`, `conformance_voltwattghi_v2`
from `ts` + `meta_up23c` + `all_uncurtailedpv_v2` (Stage 1 output).

**Run the cells in order.** Sections 1–4 are the smoke test on a single month /
single site-slice; do not skip to Section 5 until Section 4 comes back clean.

| Issue | Fixed in |
|---|---|
| R1 max(voltage) | `stage2_common.site_agg_cte` |
| R2 flex_export_detected = False | `stage2_common.meta_filter` (`exclude_flex=True`) |
| R3 / R9 AEST dates | `stage2_common.aest_month_window` + `temporal_cols` |
| R4 column naming | `build_conformance_voltvar` (thresholds unchanged) |
| R7 V-VAr curtailment zone | `build_conformance_voltvar` |
| R10 capability on S_99 | `as4777_curves.q_cap_absorbing_sql('P_kW','S_99')` |
| R11 curve keystone | both builders import, none re-implement |
| R12 AEST day/night | `stage2_common.temporal_cols` |
| R13 total_count | `build_conformance_voltwatt` (both tables agree) |
| R14 null_uncurtailed_P_count | both GHI-joined tables |
| R15 NULL not 0 | all curtailment columns |
| R16 2024 + 2025 | Section 5 |


## 0. Setup

In [1]:
import sys
import os
import time
import importlib
from pathlib import Path

# Point Python at shared/ and this Stage 2 folder.
ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT / "shared"))
sys.path.insert(0, str(ROOT / "data_calc_write" / "stage2_conformance"))

from aws_config import aq, tables, databases
from ciccada_config import SA, SAI
from stage2_common import aest_month_window

# Reload the curve keystone before reloading builders that import from it.
import as4777_curves
import build_conformance_voltvar as vv
import build_conformance_voltwatt as vw

as4777_curves = importlib.reload(as4777_curves)
vv = importlib.reload(vv)
vw = importlib.reload(vw)

DB = SAI
N_PARTS = 8

print(
    "targets:",
    "\n", vv.TARGET,
    "\n", vw.TARGET_BASIC,
    "\n", vw.TARGET_GHI,
)

print("\nVolt-VAr capability SQL:")
print(as4777_curves.q_cap_absorbing_sql("P_kW", "ac_capacity_kw"))

Using VVAR_V3 = 240.0 V (AS4777.2:2020 Australia A)
Using VW_V1 = 253.0 V (AS4777.2:2020 Australia A)
Using QCAP_P_MIN = 20% of rated apparent power (ac_capacity_kw proxy)
Using VW_V1 = 253.0 V (AS4777.2:2020 Australia A)
Using VVAR_V3 = 240.0 V (AS4777.2:2020 Australia A)
Using VW_V1 = 253.0 V (AS4777.2:2020 Australia A)
Using QCAP_P_MIN = 20% of rated apparent power (ac_capacity_kw proxy)
Using VW_V1 = 253.0 V (AS4777.2:2020 Australia A)
targets: 
 conformance_voltvar_v2 
 conformance_voltwatt_v2 
 conformance_voltwattghi_v2

Volt-VAr capability SQL:
CASE
            WHEN ac_capacity_kw <= 0 THEN 0
            WHEN abs(P_kW) <  0.2 * ac_capacity_kw THEN 0
            WHEN abs(P_kW) <= 0.6 * ac_capacity_kw
                THEN -0.44 * ac_capacity_kw
            WHEN abs(P_kW) <= 0.8 * ac_capacity_kw
                THEN -abs(P_kW) * sqrt(power(1.0 / 0.8, 2) - 1.0)
            WHEN (power(ac_capacity_kw, 2) - power(abs(P_kW), 2)) <= 0 THEN 0
            ELSE -sqrt(power(ac_capacity_kw,

In [2]:
# Connection + Stage 1 dependency check.
# Iceberg tables return nothing from DESCRIBE -- use SELECT * LIMIT 1.
aq("SELECT count(*) AS n_rows, count(DISTINCT site_id) AS n_sites "
   "FROM all_uncurtailedpv_v2", database=DB)

,n_rows,n_sites
0,467332679,14299


In [3]:
stage1_keys = aq("""
    WITH key_counts AS (
        SELECT site_id, t_stamp, count(*) AS n
        FROM all_uncurtailedpv_v2
        GROUP BY site_id, t_stamp
    )
    SELECT
        count_if(n > 1) AS duplicated_keys,
        coalesce(sum(CASE WHEN n > 1 THEN n - 1 ELSE 0 END), 0) AS excess_rows,
        max(n) AS maximum_rows_per_key
    FROM key_counts
""", database=DB)

display(stage1_keys)

duplicated_keys = int(stage1_keys["duplicated_keys"].iloc[0])
excess_rows = int(stage1_keys["excess_rows"].iloc[0])
maximum_rows = int(stage1_keys["maximum_rows_per_key"].iloc[0])

assert duplicated_keys == 0, f"Stage 1 has {duplicated_keys:,} duplicated keys"
assert excess_rows == 0, f"Stage 1 has {excess_rows:,} excess rows"
assert maximum_rows <= 1, f"Maximum rows per key is {maximum_rows}"

print("Stage 1 uniqueness check passed.")

,duplicated_keys,excess_rows,maximum_rows_per_key
0,0,0,1


Stage 1 uniqueness check passed.


In [4]:
# Stage 2 metadata cardinality check.
#
# Stage 2 joins raw telemetry to meta_up23c by circuit_id. Each eligible
# circuit must resolve to exactly one site, polarity, capacity and S_99.
metadata_diagnostics = aq("""
    WITH eligible_metadata AS (
        SELECT DISTINCT
            circuit_id,
            site_id,
            circuit_polarity,
            ac_capacity_kw,
            s_99
        FROM meta_up23c
        WHERE is_pv = true
          AND flex_export_detected = false
          AND ac_capacity_kw > 0
          AND s_99 > 0
    ),
    per_circuit AS (
        SELECT
            circuit_id,
            count(*) AS n_variants,
            count(DISTINCT site_id) AS n_sites,
            count(DISTINCT circuit_polarity) AS n_polarities,
            count(DISTINCT ac_capacity_kw) AS n_capacities,
            count(DISTINCT s_99) AS n_s99_values,
            count_if(site_id IS NULL) AS null_site_rows,
            count_if(circuit_polarity IS NULL) AS null_polarity_rows
        FROM eligible_metadata
        GROUP BY circuit_id
    )
    SELECT
        count(*) AS eligible_circuits,
        count_if(n_variants > 1) AS circuits_with_variants,
        count_if(n_sites > 1) AS circuits_with_multiple_sites,
        count_if(n_polarities > 1) AS circuits_with_conflicting_polarity,
        count_if(n_capacities > 1) AS circuits_with_conflicting_capacity,
        count_if(n_s99_values > 1) AS circuits_with_conflicting_s99,
        count_if(null_site_rows > 0) AS circuits_with_null_site,
        count_if(null_polarity_rows > 0) AS circuits_with_null_polarity
    FROM per_circuit
""", database=DB)

display(metadata_diagnostics)

,eligible_circuits,circuits_with_variants,circuits_with_multiple_sites,circuits_with_conflicting_polarity,circuits_with_conflicting_capacity,circuits_with_conflicting_s99,circuits_with_null_site,circuits_with_null_polarity
0,26397,0,0,0,0,0,0,0


In [5]:
diag = metadata_diagnostics.iloc[0]

fatal_columns = [
    "circuits_with_multiple_sites",
    "circuits_with_conflicting_polarity",
    "circuits_with_conflicting_capacity",
    "circuits_with_conflicting_s99",
    "circuits_with_null_site",
    "circuits_with_null_polarity",
]

failures = {
    column: int(diag[column])
    for column in fatal_columns
    if int(diag[column]) != 0
}

assert not failures, f"Ambiguous Stage 2 circuit metadata: {failures}"

print("Stage 2 metadata cardinality check passed.")

Stage 2 metadata cardinality check passed.


## 1. Read the SQL before running it


`preview_sql` builds the exact INSERT for one slice without executing it.


Check the AEST window: for AEST January 2024 it should read UTC partitions
`(2023,12)` and `(2024,1)`, from `2023-12-31 14:00:00` to `2024-01-31 14:00:00`.

In [6]:
print(aest_month_window(2024, 1))   # -> ('2023-12-31 14:00:00', '2024-01-31 14:00:00', [(2023,12),(2024,1)])
print(aest_month_window(2024, 7))
print(aest_month_window(2025, 12))

('2023-12-31 14:00:00', '2024-01-31 14:00:00', [(2023, 12), (2024, 1)])
('2024-06-30 14:00:00', '2024-07-31 14:00:00', [(2024, 6), (2024, 7)])
('2025-11-30 14:00:00', '2025-12-31 14:00:00', [(2025, 11), (2025, 12)])


In [7]:
print(vv.preview_sql(year=2024, month=1, n_parts=N_PARTS, part=0)[:4000])


    INSERT INTO conformance_voltvar_v2
    WITH
    
    data AS (
        SELECT
            m.site_id,
            ts.t_stamp,
            sum(ts.power * m.circuit_polarity) / 1000 AS P_kW,
            sum(ts.energy_reactive * m.circuit_polarity) / 1000 * 12 AS Q_kvar,
            max(ts.voltage)        AS V,               -- original was avg(voltage)
            max(m.ac_capacity_kw)  AS ac_capacity_kw,  -- nameplate: drives the STANDARD's curves
            max(m.s_99)            AS S_99             -- empirical: drives curtailment-limit detection
        FROM ts
        JOIN (
            SELECT circuit_id,
                max(site_id)          AS site_id,
                max(circuit_polarity) AS circuit_polarity,
                max(ac_capacity_kw)   AS ac_capacity_kw,
                max(s_99)             AS s_99
            FROM meta_up23c
            WHERE is_pv = True AND ac_capacity_kw > 0 AND s_99 > 0 AND flex_export_detected = False AND site_id % 8 = 0
            GROUP B

In [ ]:
'''
preview = vv.preview_sql(
    year=2024,
    month=1,
    n_parts=N_PARTS,
    part=0,
)

# The inherited low/moderate-P capability branches must be gone.
for obsolete_fragment in (
    "0.2 * S_99",
    "0.6 * S_99",
    "0.8 * S_99",
):
    assert obsolete_fragment not in preview, (
        f"Old capability branch remains: {obsolete_fragment}"
    )

# The generated INSERT must contain the physical S-circle.
assert "power(S_99, 2) - power(abs(P_kW), 2)" in preview

print("Volt-VAr builder is using the pure S-circle capability.")
print(preview[:4000])
'''

In [8]:
import math

S = 5.0

# Physical fixed-P Figure 2.1 boundary.
physical_checks = {
    0.0: 0.0,
    0.5: 0.0,
    1.0: -2.2,
    3.0: -2.2,
    3.5: -2.625,
    4.0: -3.0,
    4.5: -math.sqrt(4.75),
    5.0: 0.0,
    5.5: 0.0,
}

for p, expected in physical_checks.items():
    actual = as4777_curves.q_cap_absorbing(p, S)
    assert math.isclose(actual, expected, abs_tol=1e-12), (
        f"Physical boundary P={p}: expected {expected}, got {actual}"
    )

# Conformance floor after applying reactive-power priority.
response_checks = {
    0.0: 0.0,
    0.5: 0.0,
    1.0: -2.2,
    3.0: -2.2,
    3.5: -2.625,
    4.0: -3.0,
    4.5: -3.0,
    5.0: -3.0,
    5.5: -3.0,
}

for p, expected in response_checks.items():
    actual = as4777_curves.q_conformance_floor_absorbing(p, S)
    assert math.isclose(actual, expected, abs_tol=1e-12), (
        f"Conformance floor P={p}: expected {expected}, got {actual}"
    )
    print(
        f"P={p:4.1f} kW -> "
        f"conformance absorbing floor={actual:+.4f} kvar"
    )

preview = vv.preview_sql(
    year=2024,
    month=8,
    n_parts=N_PARTS,
    part=7,
)

assert "abs(P_kW) < 0.2 * ac_capacity_kw" in preview
assert "abs(P_kW) <= 0.6 * ac_capacity_kw" in preview
assert "abs(P_kW) <= 0.8 * ac_capacity_kw" in preview
assert "ELSE -0.6 * ac_capacity_kw" in preview
assert "capability_assessable" in preview

print("Physical capability and reactive-priority conformance checks passed.")

P= 0.0 kW -> conformance absorbing floor=+0.0000 kvar
P= 0.5 kW -> conformance absorbing floor=+0.0000 kvar
P= 1.0 kW -> conformance absorbing floor=-2.2000 kvar
P= 3.0 kW -> conformance absorbing floor=-2.2000 kvar
P= 3.5 kW -> conformance absorbing floor=-2.6250 kvar
P= 4.0 kW -> conformance absorbing floor=-3.0000 kvar
P= 4.5 kW -> conformance absorbing floor=-3.0000 kvar
P= 5.0 kW -> conformance absorbing floor=-3.0000 kvar
P= 5.5 kW -> conformance absorbing floor=-3.0000 kvar
Physical capability and reactive-priority conformance checks passed.


## 2. Create the empty tables

Destructive: drops and recreates. `_v2` suffix throughout.
(originals never touched)

In [ ]:
print(vv.create_table(aq, database=DB))
print(vw.create_table_basic(aq, database=DB))
print(vw.create_table_ghi(aq, database=DB))
time.sleep(5)   
tables(DB)[tables(DB)["Table"].str.contains("conformance")][["Table"]]

In [9]:
# Destructive: drops and recreates only conformance_voltvar_v2.
print(vv.create_table(aq, database=DB))

time.sleep(5)

tables(DB)[
    tables(DB)["Table"].str.contains("conformance_voltvar")
][["Table"]]

Created empty conformance_voltvar_v2


,Table
6,conformance_voltvar
7,conformance_voltvar_v2


In [10]:

TEST_SITE = 1033373679
TEST_YEAR = 2024
TEST_MONTH = 8

utc_start, utc_end, partitions = aest_month_window(
    TEST_YEAR,
    TEST_MONTH,
)

test_sql = vv._insert_sql(
    partitions=partitions,
    utc_start=utc_start,
    utc_end=utc_end,
    part_filter=f"site_id = {TEST_SITE}",
    exclude_flex=True,
)

print(
    f"Loading site {TEST_SITE}, "
    f"AEST {TEST_YEAR}-{TEST_MONTH:02d}"
)

aq(test_sql, database=DB)

print("Single-site test loaded.")


Loading site 1033373679, AEST 2024-08
Single-site test loaded.


In [11]:

site_test = aq(f"""
    SELECT
        day_night,

        sum(nonconformance_voltvar_count)
            AS nonconformance_count,

        sum(Q_adverse_count)
            AS adverse_count,

        sum(Q_inactive_count)
            AS inactive_count,

        sum(Q_significant_shortfall_count)
            AS significant_shortfall_count,

        sum(Q_near_conformant_count)
            AS near_conformant_count,

        sum(Q_major_surplus_count)
            AS major_surplus_count,

        sum(exposed_count)
            AS exposed_count,

        sum(low_power_exposed_count)
            AS low_power_exposed_count,

        sum(low_power_count)
            AS low_power_count,

        sum(total_count)
            AS assessable_count,

        sum(all_intervals_count)
            AS all_intervals_count

    FROM {vv.TARGET}

    WHERE site_id = {TEST_SITE}
      AND year = {TEST_YEAR}
      AND month = {TEST_MONTH}

    GROUP BY day_night
    ORDER BY day_night
""", database=DB)

site_test


,day_night,nonconformance_count,adverse_count,inactive_count,significant_shortfall_count,near_conformant_count,major_surplus_count,exposed_count,low_power_exposed_count,low_power_count,assessable_count,all_intervals_count
0,day,1581,0,0,0,1375,206,4376,1600,1608,2776,4384
1,night,0,0,0,0,0,0,4339,4339,4353,0,4353


In [12]:

assert (
    site_test["assessable_count"]
    + site_test["low_power_count"]
    == site_test["all_intervals_count"]
).all(), "Assessable plus low-power intervals do not equal all intervals"

assert (
    site_test["low_power_exposed_count"]
    <= site_test["low_power_count"]
).all(), "Low-power exposed count exceeds low-power count"

assert (
    site_test["low_power_exposed_count"]
    <= site_test["exposed_count"]
).all(), "Low-power exposed count exceeds exposed count"

print("Single-site interval accounting passed.")


Single-site interval accounting passed.


In [13]:
vv.validate(aq, database=DB)

Rows / sites / days per AEST month:
 year  month  n_rows  n_sites  n_days
 2024      8      62        1      31

Duplicate (year,month,day,day_night,site_id) keys (MUST be 0): 0

Coherence (all MUST be 0):
 neg_curtailment_rows  bucket_mismatch_rows  count_inversion_rows  assessment_partition_mismatch_rows  low_power_exposure_inversion_rows
                    0                     0                     0                                   0                                  0

Counterfactual coverage in the V-VAr curtailment zone:
 eligible  no_counterfactual  pct_missing
        4                  2         50.0


(   year  month  n_rows  n_sites  n_days
 0  2024      8      62        1      31,
    n_dupe_keys
 0            0,
    neg_curtailment_rows  bucket_mismatch_rows  count_inversion_rows  \
 0                     0                     0                     0   
 
    assessment_partition_mismatch_rows  low_power_exposure_inversion_rows  
 0                                   0                                  0  ,
    eligible  no_counterfactual  pct_missing
 0         4                  2         50.0)

## 3. Test one AEST month, one site slice

`parts=[0]` is 1/8 of sites for AEST January 2024. This should complete in a few minutes. 

In [ ]:
vv.run_months_voltvar(aq, database=DB, year=2024, months=[1],
                      n_parts=N_PARTS, parts=[0], exclude_flex=True)

In [ ]:
vw.run_months_basic(aq, database=DB, year=2024, months=[1],
                    n_parts=N_PARTS, parts=[0], exclude_flex=True)

In [ ]:
vw.run_months_ghi(aq, database=DB, year=2024, months=[1],
                  n_parts=N_PARTS, parts=[0], exclude_flex=True)

## 4. Validate the test

Every "MUST be 0" line must actually be 0 before you go any further.

The one to watch is **duplicate keys**. If that is non-zero, the AEST window logic has leaked and a site-day has been split across two INSERTs.

In [ ]:
vv.validate(aq, database=DB);

In [ ]:
vw.validate_basic(aq, database=DB);

In [ ]:
vw.validate_ghi(aq, database=DB);

In [ ]:
# Eyeball actual rows. 
# Check: day spans 1..31, day_night has both values,
# and P_kW_sum is positive during the day.
aq(f"""
    SELECT site_id, year, month, day, day_night,
           round(P_kW_sum, 1)                   AS P_kW_sum,
           round(nonconformance_voltvar_sum, 3) AS nonconf,
           round(curtailment_voltvar_sum, 3)    AS curtail,
           curtailment_eligible_count, null_uncurtailed_P_count,
           exposed_count, all_intervals_count, total_count
    FROM {vv.TARGET}
    ORDER BY curtailment_voltvar_sum DESC NULLS LAST
    LIMIT 15
""", database=DB)

In [ ]:
# regression test: the AEST boundary.
# Under original UTC extraction, intervals from 00:00-09:55 AEST were booked to
# the previous day. Here, day 1 of the month must contain a full AEST day --
# including its early-morning (night) intervals, which live in the PREVIOUS UTC
# month's partition. If day=1 has far fewer intervals than day=2, the window
# logic is wrong.
# NOTE: Since the data starts on 2024, we are missing the 10h from 31-DEC-2023, so day=1 will have 10 fewer intervals than day=2.
aq(f"""
    SELECT day, sum(all_intervals_count) AS intervals, count(DISTINCT site_id) AS sites
    FROM {vv.TARGET}
    WHERE year = 2024 AND month = 1 AND day IN (1, 2, 15, 30, 31)
    GROUP BY day ORDER BY day
""", database=DB)

## 5. Full load (2024 **and** 2025)

Each call is one AEST month * 8 site-slices = 8 Athena queries. 
A full year is 96 queries per table. 
Run one table at a time and check the printout.

If Athena throttles (`TooManyRequestsException`), 
drop `N_PARTS` to 4 or run `months` in two halves.

In [ ]:
# Remove all test-slice output before the production run.
# These functions drop and recreate only the _v2 Stage 2 tables.
print(vv.create_table(aq, database=DB))
print(vw.create_table_basic(aq, database=DB))
print(vw.create_table_ghi(aq, database=DB))

print("Test results cleared; empty production tables recreated.")

In [ ]:
'''
# Volt-VAr, 2024. Part 0 of Jan is already loaded from above
# rerun it and you WILL double-count. Load Jan parts 1..7:
vv.run_months_voltvar(aq, database=DB, year=2024, months=[1],
                      n_parts=N_PARTS, parts=list(range(1, N_PARTS)))
# then Feb-Dec fully:
vv.run_months_voltvar(aq, database=DB, year=2024, months=list(range(2, 13)),
                      n_parts=N_PARTS)

vv.run_months_voltvar(aq, database=DB, year=2025, months=list(range(1, 13)),
                      n_parts=N_PARTS)
'''

MONTHS = list(range(1, 13))

# Volt-VAr: complete 2024 and 2025.
for year in (2024, 2025):
    vv.run_months_voltvar(
        aq,
        database=DB,
        year=year,
        months=MONTHS,
        n_parts=N_PARTS,
        exclude_flex=True,
    )

In [ ]:
# Basic Volt-Watt: complete 2024 and 2025.
for year in (2024, 2025):
    vw.run_months_basic(
        aq,
        database=DB,
        year=year,
        months=MONTHS,
        n_parts=N_PARTS,
        exclude_flex=True,
    )

In [ ]:
# GHI-aware Volt-Watt: complete 2024 and 2025.
for year in (2024, 2025):
    vw.run_months_ghi(
        aq,
        database=DB,
        year=year,
        months=MONTHS,
        n_parts=N_PARTS,
        exclude_flex=True,
    )

## 6. Full validation

### 6.1. Volt-Var

In [22]:
vv.validate(aq, database=DB);

Rows / sites / days per AEST month:
 year  month  n_rows  n_sites  n_days
 2024      1  667804    11339      31
 2024      2  641181    11540      29
 2024      3  692436    11522      31
 2024      4  675848    11631      30
 2024      5  698418    11733      31
 2024      6  668280    11675      30
 2024      7  662586    11303      31
 2024      8  657899    11068      31
 2024      9  635615    11019      30
 2024     10  657154    11072      31
 2024     11  640374    11158      30
 2024     12  670444    11165      31
 2025      1  671797    11357      31
 2025      2  598275    11123      28
 2025      3  633925    10769      31
 2025      4  614536    10552      30
 2025      5  628434    10542      31
 2025      6  517058    10269      30
 2025      7  618363    10458      31
 2025      8  606885    10147      31
 2025      9  573473     9975      30
 2025     10  579826     9766      31
 2025     11  544404     9463      30
 2025     12  546209     9183      31

Duplicate (ye

In [ ]:
shape, dupes, coherence, cover = vv.validate(
    aq,
    database=DB,
)

In [ ]:
assert int(dupes["n_dupe_keys"].iloc[0]) == 0
assert (coherence.fillna(0) == 0).all(axis=None)

print("Full Volt-VAr rebuild passed structural validation.")

In [ ]:
aq(f"""
    SELECT
        year,
        month,

        count(*) AS rows,
        count(DISTINCT site_id) AS sites,

        sum(total_count) AS assessable_intervals,
        sum(low_power_count) AS low_power_intervals,
        sum(all_intervals_count) AS all_intervals,

        sum(total_count)
            + sum(low_power_count)
            - sum(all_intervals_count)
            AS interval_balance

    FROM {vv.TARGET}

    GROUP BY year, month
    ORDER BY year, month
""", database=DB)

### 6.2. Volt-Watt

In [23]:
vw.validate_basic(aq, database=DB);

Rows / sites / days per AEST month:
 year  month  n_rows  n_sites  n_days
 2024      1  667804    11339      31
 2024      2  641181    11540      29
 2024      3  692436    11522      31
 2024      4  675848    11631      30
 2024      5  698418    11733      31
 2024      6  668280    11675      30
 2024      7  662586    11303      31
 2024      8  657899    11068      31
 2024      9  635615    11019      30
 2024     10  657154    11072      31
 2024     11  640374    11158      30
 2024     12  670444    11165      31
 2025      1  671797    11357      31
 2025      2  598275    11123      28
 2025      3  633925    10769      31
 2025      4  614536    10552      30
 2025      5  628434    10542      31
 2025      6  517058    10269      30
 2025      7  618363    10458      31
 2025      8  606885    10147      31
 2025      9  573473     9975      30
 2025     10  579826     9766      31
 2025     11  544404     9463      30
 2025     12  546209     9183      31

Duplicate key

In [24]:
vw.validate_ghi(aq, database=DB);

Rows / sites / days per AEST month:
 year  month  n_rows  n_sites  n_days
 2024      1  667804    11339      31
 2024      2  641181    11540      29
 2024      3  692436    11522      31
 2024      4  675848    11631      30
 2024      5  698418    11733      31
 2024      6  668280    11675      30
 2024      7  662586    11303      31
 2024      8  657899    11068      31
 2024      9  635615    11019      30
 2024     10  657154    11072      31
 2024     11  640374    11158      30
 2024     12  670444    11165      31
 2025      1  671797    11357      31
 2025      2  598275    11123      28
 2025      3  633925    10769      31
 2025      4  614536    10552      30
 2025      5  628434    10542      31
 2025      6  517058    10269      30
 2025      7  618363    10458      31
 2025      8  606885    10147      31
 2025      9  573473     9975      30
 2025     10  579826     9766      31
 2025     11  544404     9463      30
 2025     12  546209     9183      31

Duplicate key

In [25]:
# R13 regression test: the two Volt-Watt tables must share a denominator.
vw.cross_check(aq, database=DB);

Basic versus GHI Volt-Watt cross-check (all MUST be 0):
 keys_missing_from_basic  keys_missing_from_ghi  total_count_mismatches  all_intervals_mismatches  power_sum_mismatches
                       0                      0                       0                         0                     0


In [26]:
# If no duplicates in the source, compare the tables directly
aq(f"""
    SELECT b.total_count AS basic_tc, g.total_count AS ghi_tc,
           g.total_count - b.total_count AS diff,
           b.all_intervals_count AS basic_all, g.all_intervals_count AS ghi_all
    FROM {vw.TARGET_BASIC} b
    JOIN {vw.TARGET_GHI} g
      ON b.site_id = g.site_id AND b.year = g.year
     AND b.month = g.month AND b.day = g.day AND b.day_night = g.day_night
    WHERE b.total_count <> g.total_count
    LIMIT 10
""", database=DB)

#Empty table = success

,basic_tc,ghi_tc,diff,basic_all,ghi_all


In [ ]:
monthly_cf_coverage = aq(f"""
    SELECT
        year,
        month,
        sum(curtailment_eligible_count) AS eligible,
        sum(null_uncurtailed_P_count) AS missing_counterfactual,
        sum(curtailment_eligible_count)
            - sum(null_uncurtailed_P_count) AS usable_counterfactual,
        round(
            100.0 * (
                sum(curtailment_eligible_count)
                - sum(null_uncurtailed_P_count)
            )
            / nullif(sum(curtailment_eligible_count), 0),
            2
        ) AS usable_pct
    FROM {vv.TARGET}
    GROUP BY year, month
    ORDER BY year, month
""", database=DB)

monthly_cf_coverage

## 7. Reconcile against original tables

Differences are **expected**. They should be fully explained by:

- flex-export sites excluded (biggest effect; ~700 sites in Stage 1)
- `max(voltage)` is >= `avg(voltage)`, so more intervals cross 240 V / 253 V
- AEST day boundaries reshuffle intervals between days
- capability clamped on `s_99`, not nameplate

If the site-count drop does **not** match the flex-export count, stop and find out why before trusting anything.

In [27]:
vv.compare_to_original(aq, database=DB, original="conformance_voltvar");

sites in conformance_voltvar_v2:  15,609
sites in conformance_voltvar: 16,147
sites dropped:       539
flex-export sites (expected explanation for the drop): 539


In [28]:
# Fleet-level headline comparison. Expect the same order of magnitude, not
# identical numbers.
aq(f"""
    SELECT 'v2' AS tbl, year,
           round(sum(nonconformance_voltvar_sum), 0)  AS nonconf_kvar,
           round(sum(curtailment_voltvar_sum), 0)     AS curtail_kw,
           sum(total_count)                           AS intervals
    FROM {vv.TARGET} GROUP BY year
    ORDER BY year
""", database=DB)

,tbl,year,nonconf_kvar,curtail_kw,intervals
0,v2,2024,265201343.0,1136.0,332234973
1,v2,2025,217011575.0,486.0,291881647


## 8. the number you cannot publish yet

`nonconformance_voltvar_red_sum` reproduces Hossein's definition:
`adverse + inactive + near_conformant` — i.e. it counts the sites that are
essentially complying and ignores the ones falling well short.

`nonconformance_voltvar_red_alt_sum` is `adverse + inactive + significant_shortfall`
— what the definition almost certainly should be.

Run this, take both numbers to Baran, and get him to confirm which range CANVAS
intended before either appears in the paper.

In [ ]:
aq(f"""
    SELECT year,
           round(sum(nonconformance_voltvar_red_sum), 0)     AS red_sum,
           sum(nonconformance_voltvar_red_count)             AS red_count,
           round(sum(Q_adverse_sum), 0)                      AS adverse,
           round(sum(Q_inactive_sum), 0)                     AS inactive,
           round(sum(Q_near_conformant_sum), 0)              AS near_conformant,
           round(sum(Q_significant_shortfall_sum), 0)        AS significant_shortfall
    FROM {vv.TARGET}
    GROUP BY year ORDER BY year
""", database=DB)

In [ ]:
aq("""
    SELECT count(*) AS n_duplicate_keys
    FROM (
        SELECT site_id, t_stamp
        FROM all_uncurtailedpv_v2
        GROUP BY site_id, t_stamp
        HAVING count(*) > 1
    )
""", database=DB)